# Modeling Workflow
### Woojin Park, Korea University
### Copyright 2026, Korea University

# Mistral Classification Sample Fine-Tuning

## Requirements

In [ ]:
# Optional dependency install for a fresh environment.
# !pip install -r ../requirements.txt

## Imports

In [ ]:
train_df, validation_df, test_df = prepare_modeling_splits(
    input_path=INPUT_PATH,
    output_dir=PROJECT_ROOT / 'data' / '03_modeling_inputs',
    run_name=RUN_NAME,
    text_column=TEXT_COLUMN,
    samples_per_class=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE,
    export_csv=True,
)

print('train:', train_df.shape)
print('validation:', validation_df.shape)
print('test:', test_df.shape)
print('')
print('Train label counts:')
print(train_df['label_str'].value_counts().sort_index())
print('')
print('Saved modeling inputs to:', MODELING_INPUT_DIR)

## Configuration

In [ ]:
# Change this value for larger experiments, for example 20000 or 40000.
SAMPLES_PER_CLASS = 1000
RANDOM_STATE = 42
TEXT_COLUMN = 'title_with_selftext_cleaned'

INPUT_PATH = PROJECT_ROOT / 'data' / '02_preprocessing_outputs' / 'final_preprocessed_df.csv'
RUN_NAME = f'sample_{SAMPLES_PER_CLASS}_per_class'
MODELING_INPUT_DIR = PROJECT_ROOT / 'data' / '03_modeling_inputs' / RUN_NAME

CLASS_NAMES = [ID_TO_LABEL[i] for i in sorted(ID_TO_LABEL)]
CLASS_NAMES
MODEL_CHECKPOINT = 'mistralai/Mistral-7B-v0.1'
MAX_LEN = 512
BATCH_SIZE = 3
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
OUTPUT_DIR = PROJECT_ROOT / 'models' / f'mistral_lora_{RUN_NAME}'

## Load, Sample, and Split Dataset

In [ ]:
train_df, validation_df, test_df = prepare_modeling_splits(
    input_path=INPUT_PATH,
    output_dir=PROJECT_ROOT / 'data' / '03_modeling_inputs',
    run_name=RUN_NAME,
    text_column=TEXT_COLUMN,
    samples_per_class=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE,
    export_csv=True,
)

print('train:', train_df.shape)
print('validation:', validation_df.shape)
print('test:', test_df.shape)
print('')
print('Train label counts:')
print(train_df['label_str'].value_counts().sort_index())
print('')
print('Saved modeling inputs to:', MODELING_INPUT_DIR)

## Convert to HuggingFace Dataset

In [ ]:
features = Features({'text': Value('string'), 'label': ClassLabel(names=CLASS_NAMES)})

def to_hf_dataset(df):
    return Dataset.from_pandas(df[['text', 'label']], preserve_index=False).cast(features)

dataset = DatasetDict({
    'train': to_hf_dataset(train_df),
    'validation': to_hf_dataset(validation_df),
    'test': to_hf_dataset(test_df),
})

dataset

## Metrics and Trainer

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0),
        'f1': f1_score(labels, preds, average='weighted'),
    }


def plot_normalized_confusion_matrix(trainer, encoded_dataset, split='test'):
    predictions = trainer.predict(encoded_dataset[split])
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = predictions.label_ids
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(cmap='Blues', values_format='.2f', ax=ax, colorbar=False)
    plt.title(f'Normalized confusion matrix ({split})')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

class WeightedCELossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss = torch.nn.CrossEntropyLoss()(logits, labels)
        return (loss, outputs) if return_outputs else loss

## Tokenization

In [ ]:
mistral_tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, add_prefix_space=True)
if mistral_tokenizer.pad_token is None:
    mistral_tokenizer.pad_token = mistral_tokenizer.eos_token

def tokenize(batch):
    return mistral_tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

encoded_dataset = dataset.map(tokenize, batched=False)
encoded_dataset.set_format('torch')
data_collator = DataCollatorWithPadding(tokenizer=mistral_tokenizer)

## LoRA Fine-Tuning

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(CLASS_NAMES),
    id2label={i: label for i, label in enumerate(CLASS_NAMES)},
    label2id={label: i for i, label in enumerate(CLASS_NAMES)},
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.config.pad_token_id = mistral_tokenizer.pad_token_id

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=['q_proj', 'v_proj'],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.001,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to='none',
)

trainer = WeightedCELossTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset['train'],
    eval_dataset=encoded_dataset['validation'],
    tokenizer=mistral_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluation

In [ ]:
validation_metrics = trainer.evaluate(encoded_dataset['validation'])
test_metrics = trainer.evaluate(encoded_dataset['test'])
print('validation:', validation_metrics)
print('test:', test_metrics)
plot_normalized_confusion_matrix(trainer, encoded_dataset, split='test')

## Save Model

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
mistral_tokenizer.save_pretrained(str(OUTPUT_DIR))
print('Saved model to:', OUTPUT_DIR)